# SOXX IBKR Historical 1-min QA — v0.1.0
**Read-only source, synthetic tests only in GitHub Actions, real data in Colab.** This notebook audits SOXX (`conId=12658194`), not SOX, TSM, ES or NQ. Start with `MAX_CHUNKS=5`; change to `0` **only** after inspecting pilot. UTC time labeling is provisional; outputs are **not research-ready**.

**Bootstrap hotfix:** first code cell validates repo origin, safely fast-forwards main, runs editable install and immediately registers `src/` on the current kernel path before importing the module.

In [ ]:
import sys, subprocess, importlib
from pathlib import Path

REPO = Path("/content/IBKR-market-data-cleaning-pipeline")
URL = "https://github.com/hh4832/IBKR-market-data-cleaning-pipeline.git"
SRC = REPO / "src"
print("Python:", sys.version)
assert sys.version_info >= (3, 11), "Python >=3.11 required"

# Safe on a freshly started runtime and on repeated Run All.
if (REPO / ".git").is_dir():
    origin = subprocess.check_output(
        ["git", "-C", str(REPO), "remote", "get-url", "origin"], text=True
    ).strip()
    assert origin.rstrip("/").removesuffix(".git") == URL.removesuffix(".git"), (
        f"Unexpected repo origin: {origin}"
    )
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only", "origin", "main"], check=True)
elif REPO.exists():
    raise RuntimeError(f"Existing path is not a Git repository: {REPO}")
else:
    subprocess.run(["git", "clone", "--depth", "1", URL, str(REPO)], check=True)

package_file = SRC / "ibkr_qc" / "__init__.py"
assert package_file.is_file(), f"Missing expected package: {package_file}"
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO}[colab]"],
    check=True,
)

# Editable installs use .pth files; an already running notebook kernel may not
# process new .pth entries until restart. Add src explicitly before importing.
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
importlib.invalidate_caches()
import ibkr_qc
from ibkr_qc.drive_io import authorized_drive, list_parquet_files
from ibkr_qc.pipeline import Config, run
print("ibkr_qc imported:", ibkr_qc.__file__)
print("Repository commit:", subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True
).strip())


In [ ]:
from pathlib import Path
SOURCE_FOLDER_ID = "1lIq9fWpZHy9RR2xkTIOeEAcmPniwXlNK"  # SOXX chunks
OUTPUT_PARENT_ID = "1k4kZ6kVjkfn99FW7AgRC6e9MDk848oVt"  # new QC output parent
MAX_CHUNKS = 5  # PILOT. Explicitly set 0 for FULL after review.
ASSUME_NAIVE_UTC = False  # Never change unless source timestamps verified as UTC.
CACHE_DIR = Path("/content/soxx_parquet_cache")
LOCAL_OUTPUT_DIR = Path("/content/soxx_qc_outputs")
print("Source:", SOURCE_FOLDER_ID, "Output parent:", OUTPUT_PARENT_ID, "Chunks:", MAX_CHUNKS)

In [ ]:
from ibkr_qc.drive_io import authorized_drive, list_parquet_files, filename_collision_audit
from ibkr_qc.pipeline import Config, run
import subprocess
service = authorized_drive()
files = list_parquet_files(service, SOURCE_FOLDER_ID)
print(f"Found {len(files)} SOXX Parquet file IDs in source folder.")
collisions = filename_collision_audit(files)
unresolved = [c for c in collisions if c["state"] != "BYTE_IDENTICAL_BY_DRIVE_MD5"]
print(f"Same-name groups: {len(collisions)}; unresolved groups: {len(unresolved)}")
for c in collisions[:10]:
    print(f"  {c['source_name']}: {c['file_count']} files, {c['state']}, IDs={c['drive_file_ids']}")
if len(collisions) > 10:
    print(f"  ... {len(collisions) - 10} additional same-name groups; see QA CSV after run")
print("First five:",[(f["name"], f["id"]) for f in files[:5]])
assert files, "No raw Parquet chunks found; check OAuth and source ID"

In [ ]:
repo = "/content/IBKR-market-data-cleaning-pipeline"
head = subprocess.check_output(["git","-C",repo,"rev-parse","HEAD"],text=True).strip()
print("Git SHA:",head)
config = Config(output_dir=LOCAL_OUTPUT_DIR,drive_source_id=SOURCE_FOLDER_ID,drive_output_id=OUTPUT_PARENT_ID,cache_dir=CACHE_DIR,max_chunks=MAX_CHUNKS,assume_naive_utc=ASSUME_NAIVE_UTC,git_sha=head)
out = run(config,service=service)
print("Local run:",out)

In [ ]:
import pandas as pd, json
info=json.loads((out/"run_info.json").read_text())
print(json.dumps(info,ensure_ascii=False,indent=2))
display(pd.read_csv(out/"daily_coverage.csv").head(30))
display(pd.read_csv(out/"chunk_quality.csv").tail(15))
print("Drive output:",info.get("drive_run_url","not uploaded"))